In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")
runpy.run_path(str(_bootstrap_path))


# Graph Hashing: Undirected and Directed Graphs

This notebook shows how `abstractgraph.hashing.hash_graph` treats small labeled graphs when they are undirected and when they are directed.

The key idea is that graph hashes are deterministic and label-aware. Undirected edges are orientation-invariant, while directed edges include source and target direction in the hash.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import networkx as nx

from abstractgraph.display import display_graph
from abstractgraph.hashing import hash_graph, hash_node

NBITS = 19


In [ ]:
def labeled_graph(directed=False, edges=(), labels=None):
    graph = nx.DiGraph() if directed else nx.Graph()
    labels = labels or {}
    for node, label in labels.items():
        graph.add_node(node, label=label)
    for u, v, label in edges:
        graph.add_edge(u, v, label=label)
    return graph


def graph_record(name, graph):
    return {
        "name": name,
        "kind": "directed" if graph.is_directed() else "undirected",
        "nodes": [(node, graph.nodes[node].get("label", "")) for node in graph.nodes()],
        "edges": [(u, v, graph.edges[u, v].get("label", "")) for u, v in graph.edges()],
        f"hash_graph(nbits={NBITS})": hash_graph(graph, nbits=NBITS),
    }


def print_records(records):
    for record in records:
        print(f"{record['name']}: {record['kind']}")
        print(f"  nodes: {record['nodes']}")
        print(f"  edges: {record['edges']}")
        print(f"  hash:  {record[f'hash_graph(nbits={NBITS})']}")
        print()


def _line_layout(graph):
    nodes = list(graph.nodes())
    if len(nodes) == 1:
        return {nodes[0]: (0.0, 0.0)}
    spacing = 1.7
    offset = spacing * (len(nodes) - 1) / 2
    return {node: (i * spacing - offset, 0.0) for i, node in enumerate(nodes)}


def _positions_for_graph(graph, layout):
    if layout is None:
        return _line_layout(graph) if len(graph.nodes()) <= 3 else None
    if all(node in layout for node in graph.nodes()):
        return {node: layout[node] for node in graph.nodes()}
    return _line_layout(graph)



def draw_hash_graphs(
    named_graphs,
    layout=None,
    *,
    size=(4.2, 2.6),
    show_edge_labels=True,
    title_hash=True,
):
    fig, axes = plt.subplots(
        1,
        len(named_graphs),
        figsize=(size[0] * len(named_graphs), size[1]),
        constrained_layout=True,
    )
    if len(named_graphs) == 1:
        axes = [axes]

    for ax, (name, graph) in zip(axes, named_graphs):
        display_graph(
            graph,
            ax=ax,
            pos=_positions_for_graph(graph, layout),
            node_labels=True,
            node_label_attr="label",
            edge_labels=show_edge_labels,
            edge_label_attr="label",
            pack_disconnected=False,
        )
        title = f"{name}\nhash={hash_graph(graph, nbits=NBITS)}" if title_hash else name
        ax.set_title(title, fontsize=12, pad=12)
        ax.set_aspect("equal")
        ax.axis("off")
    plt.show()


def draw_complex_graphs(named_graphs, layout):
    draw_hash_graphs(
        [(name, graph) for _panel, name, graph in named_graphs],
        layout=layout,
        size=(4.8, 3.0),
        show_edge_labels=False,
        title_hash=False,
    )

## Undirected edges ignore construction orientation

For an undirected `nx.Graph`, adding `(A, B)` or `(B, A)` describes the same labeled edge. The hash uses an order-independent combination for each edge's endpoints.

In [ ]:
undirected_ab = labeled_graph(
    directed=False,
    labels={0: "A", 1: "B"},
    edges=[(0, 1, "link")],
)
undirected_ba = labeled_graph(
    directed=False,
    labels={"right": "B", "left": "A"},
    edges=[("right", "left", "link")],
)

print_records([
    graph_record("A--B", undirected_ab),
    graph_record("B--A", undirected_ba),
])
draw_hash_graphs([
    ("A--B", undirected_ab),
    ("B--A", undirected_ba),
], layout={0: (-1, 0), 1: (1, 0), "left": (-1, 0), "right": (1, 0)})

assert hash_graph(undirected_ab, nbits=NBITS) == hash_graph(undirected_ba, nbits=NBITS)


## Directed edges keep orientation

For a directed `nx.DiGraph`, the outgoing and incoming sides of an edge are part of the node and graph hash. Reversing an edge changes the graph hash.

In [ ]:
directed_ab = labeled_graph(
    directed=True,
    labels={0: "A", 1: "B"},
    edges=[(0, 1, "link")],
)
directed_ba = labeled_graph(
    directed=True,
    labels={0: "A", 1: "B"},
    edges=[(1, 0, "link")],
)

print_records([
    graph_record("A->B", directed_ab),
    graph_record("B->A", directed_ba),
])
draw_hash_graphs([
    ("A->B", directed_ab),
    ("B->A", directed_ba),
], layout={0: (-1, 0), 1: (1, 0)})

assert hash_graph(directed_ab, nbits=NBITS) != hash_graph(directed_ba, nbits=NBITS)


## Directed and undirected graphs hash differently

Even when the node labels and visible edge labels match, directedness is included in `hash_graph`. This avoids treating `A--B` and `A->B` as the same structure.

In [ ]:
print_records([
    graph_record("A--B", undirected_ab),
    graph_record("A->B", directed_ab),
])
draw_hash_graphs([
    ("A--B", undirected_ab),
    ("A->B", directed_ab),
], layout={0: (-1, 0), 1: (1, 0)})

assert hash_graph(undirected_ab, nbits=NBITS) != hash_graph(directed_ab, nbits=NBITS)


## A slightly larger example

The same rule scales to larger graphs: an undirected path is not the same as a directed path, and reversing all arcs in a labeled directed path changes the hash when the labels anchor the ends.

In [ ]:
undirected_path = labeled_graph(
    directed=False,
    labels={0: "A", 1: "B", 2: "C"},
    edges=[(0, 1, "step"), (1, 2, "step")],
)
directed_path = labeled_graph(
    directed=True,
    labels={0: "A", 1: "B", 2: "C"},
    edges=[(0, 1, "step"), (1, 2, "step")],
)
reversed_path = labeled_graph(
    directed=True,
    labels={0: "A", 1: "B", 2: "C"},
    edges=[(1, 0, "step"), (2, 1, "step")],
)

print_records([
    graph_record("undirected path", undirected_path),
    graph_record("directed path", directed_path),
    graph_record("reversed directed path", reversed_path),
])
draw_hash_graphs([
    ("undirected path", undirected_path),
    ("directed path", directed_path),
    ("reversed directed path", reversed_path),
], layout={0: (-1, 0), 1: (0, 0), 2: (1, 0)})


## More complex structures

These six-node graphs give every node the same label, `A`. Both the undirected and directed pairs keep the same degree profile while changing the cross-edge structure.

In [ ]:
complex_layout = {
    1: (0.0, -1.0),
    2: (-1.0, -0.5),
    3: (-1.0, 0.5),
    4: (0.0, 1.0),
    5: (1.0, 0.5),
    6: (1.0, -0.5),
}
complex_labels = {node: "A" for node in complex_layout}
complex_a_edges = [
    (3, 4, ""),
    (4, 5, ""),
    (3, 5, ""),
    (3, 2, ""),
    (5, 6, ""),
    (2, 6, ""),
    (2, 1, ""),
    (1, 6, ""),
]
complex_b_edges = [
    (3, 4, ""),
    (4, 5, ""),
    (3, 2, ""),
    (5, 6, ""),
    (2, 1, ""),
    (1, 6, ""),
    (3, 6, ""),
    (2, 5, ""),
]
directed_complex_a_edges = complex_a_edges
directed_complex_b_edges = complex_b_edges

complex_a = labeled_graph(
    directed=False,
    labels=complex_labels,
    edges=complex_a_edges,
)
complex_b = labeled_graph(
    directed=False,
    labels=complex_labels,
    edges=complex_b_edges,
)
directed_complex_a = labeled_graph(
    directed=True,
    labels=complex_labels,
    edges=directed_complex_a_edges,
)
directed_complex_b = labeled_graph(
    directed=True,
    labels=complex_labels,
    edges=directed_complex_b_edges,
)

print_records([
    graph_record("complex graph (a)", complex_a),
    graph_record("complex graph (b)", complex_b),
])
draw_complex_graphs([
    ("(a)", "complex graph (a)", complex_a),
    ("(b)", "complex graph (b)", complex_b),
], complex_layout)

print_records([
    graph_record("directed complex graph (a)", directed_complex_a),
    graph_record("directed complex graph (b)", directed_complex_b),
])
draw_complex_graphs([
    ("(a)", "directed complex graph (a)", directed_complex_a),
    ("(b)", "directed complex graph (b)", directed_complex_b),
], complex_layout)

assert sorted(dict(complex_a.degree()).values()) == sorted(dict(complex_b.degree()).values())
assert hash_graph(complex_a, nbits=NBITS) != hash_graph(complex_b, nbits=NBITS)
assert hash_graph(directed_complex_a, nbits=NBITS) != hash_graph(directed_complex_b, nbits=NBITS)
assert hash_graph(complex_a, nbits=NBITS) != hash_graph(directed_complex_a, nbits=NBITS)


## Inspecting node-level hashes

`hash_node` includes the node label plus incident edge payloads. Directed graphs tag those payloads as `out` or `in`; undirected graphs tag them as `undirected`.

In [ ]:
for name, graph in [
    ("A--B", undirected_ab),
    ("A->B", directed_ab),
    ("B->A", directed_ba),
]:
    print(name)
    for node in graph.nodes():
        print(f"  node {node!r} label={graph.nodes[node]['label']!r}: {hash_node(node, graph)}")
    print()


## Takeaways

- Node labels and edge labels are part of the hash.
- Undirected graph edges combine endpoint information order-independently.
- Directed graph edges preserve source and target direction.
- `hash_graph(..., nbits=NBITS)` returns a bounded feature index; increase `nbits` when you need a larger hash space.